# OtakuScan Phase 3B: Google Colab GPU Workflow & VLM Pilot Experiment

This notebook provides the self-contained, reproducible GPU workflow for **OtakuScan Phase 3B**.

### Compute Architecture:
- **Compute Platform:** Google Colab Premium (NVIDIA T4, V100, L4, or A100 GPU).
- **Local Development VM:** 4 CPU cores, ~5.7 GiB RAM (reserved for development and lightweight inference).
- **Purpose of this Colab session:**
  1. Verify GPU hardware environment.
  2. Validate dataset isolation and strict zero-leakage constraints on Fold 0.
  3. Verify reproducibility and hyperparameter sensitivity of the adapted dialogue filter.
  4. **Execute a controlled Vision-Language Model (VLM) pilot experiment** (`Qwen/Qwen2-VL-2B-Instruct`) on permitted development pages, recording loading time, per-page latency, VRAM allocation, and transcription accuracy vs RapidOCR.

## 1. GPU Detection & Hardware Audit
Detect whether a GPU accelerator is active. In Google Colab, select: **Runtime > Change runtime type > Hardware accelerator > T4 GPU** (or A100).

In [ ]:
!nvidia-smi

import sys
import torch

print("=" * 60)
print("PYTORCH & CUDA ACCELERATOR CHECK")
print("=" * 60)
print(f"Python Version:  {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"Active GPU:      {gpu_name}")
    print(f"Dedicated VRAM:  {vram_gb:.2f} GB")
    print("STATUS: GPU hardware verified and ready.")
else:
    print("WARNING: No GPU detected! Please navigate to Runtime > Change runtime type and select a GPU.")
print("=" * 60)

## 2. Project Setup & Workspace Initialization
Configure the Colab workspace path and verify project directories (`dataset/`, `splits/`, `src/`).

If you uploaded the repository as a ZIP or cloned it to `/content/otakuscan`, uncomment and adjust the directory setup below.

In [ ]:
import os
from pathlib import Path

# Auto-detect Colab environment vs local execution
if 'google.colab' in sys.modules:
    # Option A: If mounted via Google Drive:
    # from google.colab import drive
    # drive.mount('/content/drive')
    # project_root = Path('/content/drive/MyDrive/otakuscan')

    # Option B: Direct repository root in /content:
    if Path('/content/otakuscan').is_dir():
        project_root = Path('/content/otakuscan')
    else:
        project_root = Path('/content')
else:
    project_root = Path('.').resolve()

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Working Directory: {Path.cwd()}")

# Verify essential project assets
assert (project_root / 'dataset').is_dir(), f"dataset/ directory missing at {project_root / 'dataset'}"
assert (project_root / 'splits' / 'dev_splits.json').is_file(), "splits/dev_splits.json missing!"
assert (project_root / 'src').is_dir(), "src/ package directory missing!"
print("PASSED: Project root, dataset, and code modules verified.")

## 3. Install Colab Dependencies
Install required dependencies for RapidOCR inference, transformers VLM pipeline, and vision processing.

In [ ]:
!pip install -q rapidocr-onnxruntime pillow numpy scipy "transformers>=4.45.0" accelerate qwen-vl-utils

## 4. Strict Leakage Prevention & Split Verification
Enforce strict dataset integrity:
1. `dataset/` is strictly read-only.
2. Training must use ONLY Fold 0 training sequences (64 sequences).
3. Fold 0 validation sequences (16 sequences) must NEVER be used to construct training examples.
4. Test sequences (15 sequences) must remain completely unaccessed.

In [ ]:
import json

with open(project_root / 'splits' / 'dev_splits.json', 'r', encoding='utf-8') as f:
    splits_data = json.load(f)

train_ids = set(splits_data['kfold_5'][0]['train'])
val_ids = set(splits_data['kfold_5'][0]['val'])

print(f"Fold 0 Training Sequences:   {len(train_ids)}")
print(f"Fold 0 Validation Sequences: {len(val_ids)}")

# Check for sequence overlap
overlap = train_ids.intersection(val_ids)
assert len(overlap) == 0, f"CRITICAL FAILURE: Sequence leakage detected! Overlapping IDs: {overlap}"
print("PASSED: Zero sequence leakage between training and validation sets.")

## 5. Verify & Retrain Adapted Classifiers
Verify the dialogue filter and narration classifier weights on Fold 0 training sequences.

**Architectural Note:** Our production dialogue filter is an L2-regularized linear classifier over 20 geometric and lexical features. It trains in 0.57s on CPU, evaluates in <0.1 ms per page, and requires zero heavy deep-learning serving infrastructure on the local VM. Here we verify exact reproducibility and explore hyperparameter sensitivity across $L_2$ penalties.

In [ ]:
from colab.train_classifier_colab import run_colab_training

train_summary = run_colab_training(
    dataset_root=project_root / 'dataset',
    splits_file=project_root / 'splits' / 'dev_splits.json',
    output_dir=project_root / 'models',
    run_grid_search=True,
)

print("\n--- Verification Summary ---")
print(f"Training samples: {train_summary['train_samples']}")
print(f"Weights saved to: {train_summary['filter_weights_path']}")
print(f"Grid search configurations evaluated: {len(train_summary['grid_results'])}")

## 6. Vision-Language Model (VLM) Pilot Experiment
Run a controlled pilot evaluating `Qwen/Qwen2-VL-2B-Instruct` against RapidOCR on 3 permitted Fold 0 training pages.

The pilot measures:
- Model checkpoint loading time
- GPU memory consumption (Peak VRAM allocated)
- Per-page inference latency (GPU vs RapidOCR CPU)
- Transcription token precision, recall, and F1 against ground truth dialogue
- Error analysis on sound effects, multi-column reading order, and narration captions

In [ ]:
from colab.vlm_manga_transcription_experiment import run_pilot_experiment

pilot_output_file = project_root / 'logs' / 'colab_vlm_pilot_results.json'

pilot_results = run_pilot_experiment(
    project_root=project_root,
    model_id="Qwen/Qwen2-VL-2B-Instruct",
    n_pages=3,
    output_json=pilot_output_file,
)

## 7. Comparative Audit & Results Inspection
Inspect the side-by-side performance of RapidOCR vs Qwen2-VL across the pilot pages.

In [ ]:
print("=" * 78)
print(f"{'Page':<12} {'Engine':<18} {'Latency':<12} {'Peak VRAM':<14} {'Token F1':<12}")
print("=" * 78)

for p in pilot_results['per_page_results']:
    p_name = p['page_name']
    # RapidOCR
    ocr_lat = f"{p['rapidocr']['latency_seconds']}s"
    ocr_f1 = f"{p['rapidocr']['token_match']['token_f1']:.4f}"
    print(f"{p_name:<12} {'RapidOCR':<18} {ocr_lat:<12} {'< 65 MB':<14} {ocr_f1:<12}")
    
    # VLM
    vlm_info = p['vlm']
    if vlm_info.get('status') == 'success':
        vlm_lat = f"{vlm_info['latency_seconds']}s"
        vlm_vram = f"{vlm_info['peak_vram_mb']:.1f} MB"
        vlm_f1 = f"{vlm_info['token_match']['token_f1']:.4f}"
        print(f"{p_name:<12} {'Qwen2-VL-2B':<18} {vlm_lat:<12} {vlm_vram:<14} {vlm_f1:<12}")
    else:
        print(f"{p_name:<12} {'Qwen2-VL-2B':<18} {'N/A':<12} {'N/A':<14} {vlm_info.get('status', 'skipped'):<12}")
    print("-" * 78)

print("\nPILOT LOG SAVED AT:", pilot_output_file)
print("Please download logs/colab_vlm_pilot_results.json and share the recorded metrics.")